In [ ]:
# Analyze top X dependents of pymatgen

In [ ]:
from pathlib import Path

CSV_FILE = "pymatgen_dependents_with_stars.csv"
DEST_DIR = Path("dependent-repos")
TOP_X: int = 20

In [ ]:
# Clone the top X repos

import csv
import subprocess

DEST_DIR.mkdir(parents=True, exist_ok=True)

with open(CSV_FILE, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)[:TOP_X]

for row in rows:
    repo_name = row["owner_repo"].split("/")[-1]
    repo_url = row["repo_url"]

    dest_path = DEST_DIR / repo_name
    if dest_path.is_dir():
        print(f"✅ Skipping {repo_name} (already cloned)")
        continue

    print(f"⬇️  Cloning {repo_name} ...")
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", repo_url, str(dest_path)],
            check=True,
        )
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to clone {repo_name}: {e}")
    else:
        print(f"✅ Done: {repo_name}")

✅ Skipping materials_discovery (already cloned)
✅ Skipping Uni-Mol (already cloned)
✅ Skipping AIRS (already cloned)
✅ Skipping mat2vec (already cloned)
✅ Skipping megnet (already cloned)
✅ Skipping mattersim (already cloned)
✅ Skipping dscribe (already cloned)
✅ Skipping matgl (already cloned)
✅ Skipping PaddleScience (already cloned)
✅ Skipping PyXtal (already cloned)
✅ Skipping DeepH-pack (already cloned)
✅ Skipping pymatviz (already cloned)
✅ Skipping matgenb (already cloned)
✅ Skipping all-atom-diffusion-transformer (already cloned)
✅ Skipping materials (already cloned)
✅ Skipping gptchem (already cloned)
✅ Skipping quacc (already cloned)
✅ Skipping doped (already cloned)
✅ Skipping matbench-discovery (already cloned)
✅ Skipping matsciml (already cloned)


In [ ]:
# Manually build an exclude map for each repo

from dataclasses import dataclass


@dataclass
class RepoConfig:
    exclude: list[str] | None = None


repos = sorted([p.name for p in DEST_DIR.iterdir() if p.is_dir()])

# Copiable dict to avoid putting everything manually
print("REPO_PATH_MAP: dict[str, RepoConfig] = {")
for name in repos:
    print(f'    "{name}": RepoConfig(exclude=["tests", "docs", ".github"]),')
print("}")

REPO_PATH_MAP: dict[str, RepoConfig] = {
    "AIRS": RepoConfig(exclude=["tests", "docs"]),
    "DeepH-pack": RepoConfig(exclude=["tests", "docs"]),
    "PaddleScience": RepoConfig(exclude=["tests", "docs"]),
    "PyXtal": RepoConfig(exclude=["tests", "docs"]),
    "Uni-Mol": RepoConfig(exclude=["tests", "docs"]),
    "all-atom-diffusion-transformer": RepoConfig(exclude=["tests", "docs"]),
    "doped": RepoConfig(exclude=["tests", "docs"]),
    "dscribe": RepoConfig(exclude=["tests", "docs"]),
    "gptchem": RepoConfig(exclude=["tests", "docs"]),
    "mat2vec": RepoConfig(exclude=["tests", "docs"]),
    "matbench-discovery": RepoConfig(exclude=["tests", "docs"]),
    "materials": RepoConfig(exclude=["tests", "docs"]),
    "materials_discovery": RepoConfig(exclude=["tests", "docs"]),
    "matgenb": RepoConfig(exclude=["tests", "docs"]),
    "matgl": RepoConfig(exclude=["tests", "docs"]),
    "matsciml": RepoConfig(exclude=["tests", "docs"]),
    "mattersim": RepoConfig(exclude=["tests", "docs"]),
    "megnet": RepoConfig(exclude=["tests", "docs"]),
    "pymatviz": RepoConfig(exclude=["tests", "docs"]),
    "quacc": RepoConfig(exclude=["tests", "docs"]),
}

REPO_PATH_MAP: dict[str, RepoConfig] = {
    "AIRS": RepoConfig(exclude=["tests", "docs", ".github"]),
    "DeepH-pack": RepoConfig(exclude=["tests", "docs", ".github"]),
    "PaddleScience": RepoConfig(exclude=["tests", "docs", ".github"]),
    "PyXtal": RepoConfig(exclude=["tests", "docs", ".github"]),
    "Uni-Mol": RepoConfig(exclude=["tests", "docs", ".github"]),
    "all-atom-diffusion-transformer": RepoConfig(exclude=["tests", "docs", ".github"]),
    "doped": RepoConfig(exclude=["tests", "docs", ".github"]),
    "dscribe": RepoConfig(exclude=["tests", "docs", ".github"]),
    "gptchem": RepoConfig(exclude=["tests", "docs", ".github"]),
    "mat2vec": RepoConfig(exclude=["tests", "docs", ".github"]),
    "matbench-discovery": RepoConfig(exclude=["tests", "docs", ".github"]),
    "materials": RepoConfig(exclude=["tests", "docs", ".github"]),
    "materials_discovery": RepoConfig(exclude=["tests", "docs", ".github"]),
    "matgenb": RepoConfig(exclude=["tests", "docs", ".github"])

In [ ]:
# Install api-analyzer
api_analyzer_path = Path().resolve() / "api-analyzer"
!uv pip install -e "{api_analyzer_path}"

Using Python 3.12.11 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Resolved 1 package in 17ms                                           
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/figs/pmg-ap
   Building api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/figs/pmg-ap
      Built api-analyzer @ file:///Users/yang/developer/pymatgen-2-paper/figs/pmg-ap
Prepared 1 package in 343ms                                              
Uninstalled 1 package in 0.62ms
Installed 1 package in 1ms (from file:///Users/yang/develope
 ~ api-analyzer==0.1.0 (from file:///Users/yang/developer/pymatgen-2-paper/figs/pmg-api-usage-dependent-dependency/api-analyzer)


In [ ]:
from api_analyzer import analyze_paths

results: dict[str, dict[str, dict[str, int]]] = {}

for repo_name, cfg in REPO_PATH_MAP.items():
    repo_path = DEST_DIR / repo_name
    if not repo_path.exists():
        print(f"⚠️  Skipping {repo_name}: path not found")
        continue

    print(f"🔍 Analyzing {repo_name} for pymatgen usage...")

    aliases, usage = analyze_paths(
        repo_path,
        package="pymatgen",  # analyze dependent of pmg usage
        exclude=cfg.exclude or [],
    )

    results[repo_name] = {
        "aliases": aliases,
        "usage": usage,
    }

print("✅ Analysis finished.")

🔍 Analyzing AIRS for pymatgen usage...
⚠️  Skipping dependent-repos/AIRS/OpenProt/LatentDiff/LatentDiff/data/gen_data_for_diffusion.ipynb (AST parse error: invalid syntax (<unknown>, line 48))
⚠️  Skipping dependent-repos/AIRS/OpenProt/LatentDiff/LatentDiff/ProteinMPNN/colab_notebooks/quickdemo_wAF2.ipynb (AST parse error: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (<unknown>, line 252))
⚠️  Skipping dependent-repos/AIRS/OpenMI/EquiGX/EquiGX/SCOP.ipynb (AST parse error: invalid syntax (<unknown>, line 17))
⚠️  Skipping dependent-repos/AIRS/OpenMI/EquiGX/EquiGX/BioLip.ipynb (AST parse error: invalid decimal literal (<unknown>, line 2446))
🔍 Analyzing DeepH-pack for pymatgen usage...
🔍 Analyzing PaddleScience for pymatgen usage...
⚠️  Skipping dependent-repos/PaddleScience/jointContribution/IJCAI_2024/bju/download_dataset.ipynb (AST parse error: unexpected indent (<unknown>, line 12))
⚠️  Skipping dependent-repos/PaddleScience/jointCo

In [ ]:
import plotly.graph_objects as go
from collections import defaultdict

# Illegal or deprecated top-level pymatgen APIs (by `megnet`)
ILLEGAL_PMGMODS = {"Structure", "Molecule", "MPRester", "Composition"}

# Consistent pmg submodule color mapping
PMG_COLORS = {
    "core": "#3B9C9C",
    "analysis": "#6C5B7B",
    "io": "#355C7D",
    "entries": "#99B898",
    "ext": "#E84A5F",
    "electronic_structure": "#F67280",
    "transformations": "#C06C84",
    "phonon": "#A8E6CF",
    "symmetry": "#FFD3B6",
    "optimization": "#FFAAA5",
    "util": "#FF8B94",
}
DEFAULT_PMG_COLOR = "#A0A0A0"


def hex_to_rgba(hex_color: str, alpha: float = 0.5) -> str:
    """Convert hex color like '#F67280' to rgba(246,114,128,0.5)."""
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i : i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


# Aggregate usage by repo and pymatgen submodule
flows: list[tuple[str, str, int]] = []  # (repo, submodule, count)

for repo, result in results.items():
    usage = result["usage"]
    sub_counts = defaultdict(int)

    for api, count in usage.items():
        parts = api.split(".")
        if not parts or parts[0] != "pymatgen":
            continue

        # Drop any illegal top-level access, e.g. "pymatgen.Structure"
        if len(parts) >= 2 and parts[1] in ILLEGAL_PMGMODS:
            continue

        # Extract the submodule name (e.g. "pymatgen.core.Structure" → "core")
        submod = parts[1] if len(parts) >= 2 else None
        if submod:
            sub_counts[submod] += count

    for submod, total in sub_counts.items():
        flows.append((repo, submod, total))

# Build Sankey input data
repos = sorted({r for r, _, _ in flows})
subs = sorted({s for _, s, _ in flows})

labels = repos + [f"pymatgen.{s}" for s in subs]
label_to_idx = {lbl: i for i, lbl in enumerate(labels)}

sources = [label_to_idx[r] for r, s, _ in flows]
targets = [label_to_idx[f"pymatgen.{s}"] for _, s, _ in flows]
values = [v for _, _, v in flows]

# Node colors
node_colors = []
for label in labels:
    if label in repos:
        node_colors.append("#F7B267")  # orange for downstream repos
    else:
        sub = label.replace("pymatgen.", "")
        node_colors.append(PMG_COLORS.get(sub, DEFAULT_PMG_COLOR))

# Link colors (RGBA)
link_colors = []
for _, sub, _ in flows:
    color = PMG_COLORS.get(sub, DEFAULT_PMG_COLOR)
    link_colors.append(hex_to_rgba(color, 0.5))  # 50% transparency

# Sankey diagram
fig = go.Figure(
    go.Sankey(
        arrangement="snap",
        node=dict(
            label=labels,
            pad=20,
            thickness=20,
            color=node_colors,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors,
        ),
    )
)

fig.update_layout(
    title_text="Downstream Repositories' Usage of pymatgen Submodules",
    font=dict(size=12),
    height=700,
    width=1100,
)


fig.show()
fig.write_image(
    f"{Path().cwd().parents[1]}/paper/figs/dependent-usage-of-pmg.svg",
    format="svg",
    scale=2,
)